# Import packages

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import itertools

# Load dataset

In [ ]:
# Load the dataset
dataset = pd.read_csv("dataset.csv")
# Shuffle the rows but make sure to set the seed to get reproducible results
dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)
# We use a 90-10 train/test ratio so we divide by 10. An 80-20 would mean division by 5
test_size = len(dataset) // 10
# We cannot just multiply test_size by 9, so we must subtract the test size from the total size
train_size = len(dataset) - test_size
# These are the two target names
targets = ["pXC50_3D7", "PCT_IHB_3D7"]

In [ ]:
# Get ONLY the numeric features and make sure they exist in the dataset to avoid errors later.
x = dataset.drop(targets, axis=1)
# Get the target variables
y = dataset[targets]

# Helper Functions

In [ ]:
def save_histogram(data, xlabel, ylabel, title, filename):
    """Plots and saves a histogram.

    Args:
        data: The data to be plotted.
        xlabel: The x-axis label.
        ylabel: The y-axis label.
        title: The title of the plot.
        filename: The name of the file.
    """
    plt.hist(data)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.savefig(filename)
    plt.show()

def backward_elimination(_x, _y, k):
    """Uses backward elimination to reduce the dataset to k features.

    Args:
        _x: Dataframe of features.
        _y: Dataframe of labels.
        k: Number of features to keep.

    Returns:
        statsmodels.base.model.Model(): The final model that has been reduced.
        list: List of selected features.
    """

    # Make a copy of the X argument to avoid changing the original dataframe
    _x = _x.copy()
    # Add the intercept column
    _x = sm.add_constant(_x)

    # Loop until k features remain
    while len(_x.columns) - 1 > k:
        # Fit the model on the provided data
        _model = sm.GLM(_y, _x, family=sm.families.Gamma(link=sm.families.links.Log())).fit()
        # Calculate all p-values
        pvals = _model.pvalues.drop('const')
        # Get the name of the worst feature (largest p-value)
        worst_feature = pvals.idxmax()
        # Drop the worst feature
        _x.drop(columns=[worst_feature], inplace=True)

    # Fit a fresh model with the chosen features
    final_model = sm.GLM(_y, _x, family=sm.families.Gamma(link=sm.families.links.Log())).fit()
    return final_model, _x.columns.tolist()

def remove_correlated(df, corr_dict, threshold=0.8):
    """Removes highly correlated features from a dataframe.

    Args:
        df: Dataframe of features.
        corr_dict: Dictionary of correlation coefficients. Must be in format Pair : coefficient.
        threshold: Threshold for removal (default 0.8).

    Returns:
        pd.DataFrame: Cleaned dataframe.
    """

    # Initialize the features to keep with everything
    to_keep = set(df.columns)
    # Calculate the variances of each column
    variances = df.var()

    # Loop over all feature pairs
    for (feat1, feat2), corr_val in corr_dict.items():
        # Check if the correlation coefficient is above threshold, and make sure both features are in the list to avoid double-deletion
        if corr_val > threshold and feat1 in to_keep and feat2 in to_keep:
            # Delete the variable with the higher variance
            if variances[feat1] >= variances[feat2]:
                to_keep.remove(feat2)
            else:
                to_keep.remove(feat1)

    return df[list(to_keep)]

def accuracy_score(real, predicted, factor: float):
    """Calculate accuracy within a percentage tolerance.

    Args:
        real: Array-like of true values
        predicted: Array-like of predicted values
        factor: Tolerance factor (0.1 for 10% tolerance)

    Returns:
        list: Boolean list where True indicates prediction within tolerance

    Example:
        >>> accuracy_score([10, 20], [9, 22], 0.1)
        {False: 1, True: 1}
    """

    # Calculate the absolute error
    differences = abs(real - predicted)
    # Check to make sure it's within the provided limit
    within_limit = list(differences < factor * real)
    # Count how many predictions are within the limit and how many are not
    result = np.unique(within_limit, return_counts=True)
    # Convert the results to standard data types
    result = {bool(key): int(value) for key, value in zip(result[0], result[1])}
    # Make sure both keys are present to prevent errors when calculating the accuracy score
    if True not in result:
        result[True] = 0
    if False not in result:
        result[False] = 0

    return result

def feature_scaling(df):
    """Scales all features of the given df to be more understandable

    Args:
        df: Input dataframe

    Returns:
        pandas.DataFrame: DataFrame with all features scaled
    """

    for col in df.columns:
        if abs(df[col]).max() == 0:
            continue

        while abs(df[col]).max() >= 10:
            df[col] /= 10

        while abs(df[col]).max() <= 1:
            df[col] *= 10

    return df

## Histogram before transformation

In [ ]:
# Make histograms of the two target variables
save_histogram(y[targets[0]], "Potency Values Against the 3D7 Strain", "Count", "Distribution of pIC50_3D7", "figs/hist_pic50_orig.png")
save_histogram(y[targets[1]], "Inhibition Percentage Against the 3D7 Strain", "Count", "Distribution of PCT_IHB_3D7", "figs/hist_pct_ihb_orig.png")

# Transform data

In [ ]:
# Scale all features
x = feature_scaling(x)

# Split the input dataframe into train and test
x_train = x[:train_size]
x_test = x[train_size:]

# We have to add a constant column so the constant is calculated during backward elimination
x_test = sm.add_constant(x_test)

# Split the target dataframe into train and test as well
y_train = y[:train_size][targets]
y_test = y[train_size:][targets]

# Get the min/max for the two target variables
train_min_target0 = min(y_train[targets[0]])
train_max_target1 = max(y_train[targets[1]])

# Transform the target variables to make them as close to 0 as possible for gamma regression to work
# We are trying to make both of them right skewed as shown in the plots below
y_train[targets[0]] = y_train[targets[0]] - train_min_target0 + 0.01
y_train[targets[1]] = train_max_target1 - y_train[targets[1]] + 0.01

# Histogram after transformation

In [ ]:
# Make histograms of the two target variables.
save_histogram(y_train[targets[0]], "Potency Values Against the 3D7 Strain", "Count", "Distribution of pIC50_3D7 After Transformation", "figs/hist_pic50_shift.png")
save_histogram(y_train[targets[1]], "Inhibition Percentage Against the 3D7 Strain", "Count", "Distribution of PCT_IHB_3D7 After Transformation", "figs/hist_pct_ihb_shift.png")

# Data Cleaning and Preprocessing

In [ ]:
# Get the list of all pairs of features
feature_pairs = itertools.combinations(x_train.columns, 2)
# Calculate the correlation coefficient between each pair of features
correlation_matrix = {c : float(abs(np.corrcoef(x_train[c[0]], x_train[c[1]])[0][1])) for c in feature_pairs}
# Remove the highly correlated features, we are using 0.8 as the threshold here
x_train = remove_correlated(x_train, correlation_matrix)

# Backward Elimination, Training, and Scoring for pIC50_3D7

In [ ]:
# Train a reduced model that includes ONLY the most significant 7 features
model, selected_features = backward_elimination(x_train, y_train[targets[0]], 7)
# Predict with the original scale, we can do this by applying the inverse of the transformation
y_pred = model.predict(x_test[selected_features]) + train_min_target0 - 0.01
# Print the summary shown below
print(model.summary())
print("Model AIC:", model.aic)
print()

# Change this list to change what threshold values the accuracy is calculated for
for i in [5, 10, 15, 20, 25]:
    # Get the dictionary of how many are valid and how many are invalid.
    accuracy_results = accuracy_score(real=y_test[targets[0]],
                                      predicted=y_pred,
                                      factor=i / 100)

    # We calculate the final accuracy score by calculating the ratio of predictions within the threshold to the total number of predictions.
    final_score = accuracy_results[True] / (accuracy_results[True] + accuracy_results[False])
    print(f"Accuracy within {i}%: {int(final_score * 100)}%")

# Backward Elimination, Training, and Scoring for PCT_IHB_3D7

In [ ]:
# Just as before, we train the model and reduce it to 7 features
model, selected_features = backward_elimination(x_train, y_train[targets[1]], 7)
# This time, the inverse calculation is slightly different, but the concept is the same.
y_pred = train_max_target1 - model.predict(x_test[selected_features]) + 0.01
# Print the summary again.
print(model.summary())
print("Model AIC:", model.aic)
print()

# Everything below works the same as above
for i in [5, 10, 15, 20, 25]:
    accuracy_results = accuracy_score(real=y_test[targets[1]],
                                      predicted=y_pred,
                                      factor=i / 100)

    final_score = accuracy_results[True] / (accuracy_results[True] + accuracy_results[False])
    print(f"Accuracy within {i}%: {int(final_score * 100)}%")